# Mean May–Oct precipitation change, 30–60°N: IFS, ICON and the CMIP6 models (2035–2049 vs 2021–2034)

Version of `precip_total_NH_future.ipynb` with CESM2 replaced by the CMIP6 models. IFS and ICON are unchanged (HEALPix grids with equal-area cells, so their unweighted means are area means).

**CMIP6:** the same 15 models and members as the other CMIP6 notebooks (first ssp370 member of each model, the members of the `precip_cover_<model>_ssp370_<member>_30-60N.nc` files; CanESM5 p2). Daily `pr` from cmip6-ng, 30–60°N, May–Oct, summed per year (m). Land is `sftlf > 0`, sea `sftlf == 0` (BCC-CSM2-MR uses the `FALLBACK_MASK_MODEL` mask), with cos(lat) area weights. Relative change = (2035–2049 − 2021–2034) / (2021–2034) × 100 per model; the CMIP6 mean is the mean of the model changes (equal model weight). The yearly totals of each model are cached in `CMIP6_CACHE_DIR`, so only the first run reads the daily files.

In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D

from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [3]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 1
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 64, number of workers: 1, processes: False


In [4]:
latitude = [30,60]


In [5]:
ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_2021-2049_30-60N_precip_total_sum.nc')['tp']
icon_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle4_2021-2049_30-60N_precip_total_sum.nc')['pr']
#ifs_cycle4 = ifs_cycle4.isel(year=slice(0,30))
#ifs_cycle4 = ifs_cycle4.stack(sample=("year", "time"))
#icon_cycle4 = icon_cycle4.isel(year=slice(0,5))
#icon_cycle4 = icon_cycle4.stack(sample=("year", "time"))



In [6]:
print(ifs_cycle4)

<xarray.DataArray 'tp' (year: 29, value: 574808)> Size: 133MB
[16669432 values with dtype=float64]
Coordinates:
    lon      (value) float64 5MB ...
    lat      (value) float64 5MB ...
Dimensions without coordinates: year, value


In [7]:
print(icon_cycle4)

<xarray.DataArray 'pr' (year: 29, cell: 574808)> Size: 133MB
[16669432 values with dtype=float64]
Coordinates:
    lon      (cell) float64 5MB ...
    lat      (cell) float64 5MB ...
Dimensions without coordinates: year, cell


In [8]:
land_sea_ifs_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/30-60N_ifs_hpx512_lsm.nc')['lsm']
land_sea_icon_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/30-60N_ICON_ngc4008_ocean_fraction_surface_zoom9.nc')['ocean_fraction_surface']


ifs_cycle4_land = ifs_cycle4.where(land_sea_ifs_zoom9 > 0, np.nan)
icon_cycle4_land = icon_cycle4.where(land_sea_icon_zoom9 < 1, np.nan)


ifs_cycle4_sea = ifs_cycle4.where(land_sea_ifs_zoom9 == 0, np.nan)
icon_cycle4_sea = icon_cycle4.where(land_sea_icon_zoom9 == 1, np.nan)



In [9]:
ifs_hist_land = ifs_cycle4_land.isel(year=slice(0,14))
ifs_fut_land = ifs_cycle4_land.isel(year=slice(14,29))
icon_hist_land = icon_cycle4_land.isel(year=slice(0,14))
icon_fut_land = icon_cycle4_land.isel(year=slice(14,29))


In [10]:
ifs_hist_sea = ifs_cycle4_sea.isel(year=slice(0,14))
ifs_fut_sea = ifs_cycle4_sea.isel(year=slice(14,29))
icon_hist_sea = icon_cycle4_sea.isel(year=slice(0,14))
icon_fut_sea = icon_cycle4_sea.isel(year=slice(14,29))


In [11]:
ifs_hist_land_mean = ifs_hist_land.mean() / 1000
ifs_fut_land_mean = ifs_fut_land.mean() / 1000
icon_hist_land_mean = icon_hist_land.mean() / 1000
icon_fut_land_mean = icon_fut_land.mean() / 1000


In [12]:
ifs_hist_sea_mean = ifs_hist_sea.mean() / 1000
ifs_fut_sea_mean = ifs_fut_sea.mean() / 1000
icon_hist_sea_mean = icon_hist_sea.mean() / 1000
icon_fut_sea_mean = icon_fut_sea.mean() / 1000


In [13]:
ifs_land_diff = ((ifs_fut_land_mean - ifs_hist_land_mean) / ifs_hist_land_mean) * 100
icon_land_diff = ((icon_fut_land_mean - icon_hist_land_mean) / icon_hist_land_mean) * 100


In [14]:
ifs_sea_diff = ((ifs_fut_sea_mean - ifs_hist_sea_mean) / ifs_hist_sea_mean) * 100
icon_sea_diff = ((icon_fut_sea_mean - icon_hist_sea_mean) / icon_hist_sea_mean) * 100


In [15]:
print(ifs_land_diff)

<xarray.DataArray 'tp' ()> Size: 8B
array(-3.01335571)


In [16]:

print('land area mean precip diff for ifs cycle4 is ' + str(ifs_land_diff.values) + '%')
print('land area mean precip diff for icon cycle4 is ' + str(icon_land_diff.values) + '%')



land area mean precip diff for ifs cycle4 is -3.0133557131956064%
land area mean precip diff for icon cycle4 is -4.871700143190236%


In [17]:
print('sea area mean precip diff for ifs cycle4 is ' + str(ifs_sea_diff.values) + '%')
print('sea area mean precip diff for icon cycle4 is ' + str(icon_sea_diff.values) + '%')



sea area mean precip diff for ifs cycle4 is 2.1934291579030276%
sea area mean precip diff for icon cycle4 is -0.41027065730135875%


## CMIP6 models

In [18]:
import re

CMIP6_PR_DIR    = '/net/atmos/data/cmip6-ng/pr/day/native'                  # pr_day_<model>_ssp370_<member>_native.nc
CMIP6_SFTLF_DIR = '/net/atmos/data/cmip6-ng/sftlf/fx/native'
CMIP6_PC_DIR    = '/home/jwille/xenon_data/NextGEMS/mid_lat_precip/CMIP6'    # precip_cover files: members used in the analysis
CMIP6_CACHE_DIR = '/home/jwille/xenon_data/NextGEMS/mid_lat_precip/CMIP6_total'   # yearly May-Oct totals per model
CMIP6_OUT_CSV   = '/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/cmip6_precip_diff_30-60N.csv'
CMIP6_EXCLUDE   = []                          # e.g. ['BCC-CSM2-MR']
FALLBACK_MASK_MODEL = 'MPI-ESM1-2-HR'         # mask for models without an sftlf file (BCC-CSM2-MR)
MONTHS  = [5, 6, 7, 8, 9, 10]
YEARS   = (2021, 2049)
PERIOD_HIST, PERIOD_FUT = (2021, 2034), (2035, 2049)   # = isel(year=slice(0,14)) and slice(14,29) above
UNIT_FACTORS = {'kg m-2 s-1': 86400, 'kg/m2/s': 86400, 'mm s-1': 86400, 'm s-1': 86400 * 1000, 'm/s': 86400 * 1000}   # -> mm/day


def member_sort_key(member):
    return [int(n) for n in re.findall(r'\d+', member)]


def nearest_regrid(src, lat, lon):
    """Nearest-neighbour match of a lat/lon field to the given lat/lon (lon periodic)."""
    ilat = np.abs(src.lat.values[:, None] - np.asarray(lat)[None, :]).argmin(axis=0)
    dlon = (src.lon.values[:, None] - np.asarray(lon)[None, :] + 180) % 360 - 180
    ilon = np.abs(dlon).argmin(axis=0)
    return src.values[np.ix_(ilat, ilon)]


def find_sftlf(model):
    pat = re.compile(rf'^sftlf_fx_{re.escape(model)}_(?P<exp>[^_]+)_(?P<mem>[^_]+)_[^_]+\.nc$')
    cands = [(m['exp'] != 'historical', member_sort_key(m['mem']), f)
             for f in os.listdir(CMIP6_SFTLF_DIR) if (m := pat.match(f))]
    return os.path.join(CMIP6_SFTLF_DIR, sorted(cands)[0][2]) if cands else None


def sftlf_on(model, lat, lon):
    path = find_sftlf(model)
    src = xr.open_dataset(path or find_sftlf(FALLBACK_MASK_MODEL))['sftlf'].load()
    return nearest_regrid(src, lat, lon), (os.path.basename(path) if path else f'{FALLBACK_MASK_MODEL} (fallback)')


def yearly_totals(model, member):
    """May-Oct precipitation total (m) per year and grid cell at 30-60N, 2021-2049; cached."""
    cache = os.path.join(CMIP6_CACHE_DIR, f'pr_total_MJJASO_{model}_ssp370_{member}_{YEARS[0]}-{YEARS[1]}_30-60N.nc')
    if os.path.exists(cache):
        return xr.open_dataset(cache)['pr_total'].load()
    path = os.path.join(CMIP6_PR_DIR, f'pr_day_{model}_ssp370_{member}_native.nc')
    da = xr.open_dataset(path, use_cftime=True)['pr']
    units = da.attrs.get('units', '').strip()
    if units not in UNIT_FACTORS:
        raise ValueError(f"{model}: unknown pr units '{units}'")
    lat = da.lat.values
    da = da.isel(lat=np.where((lat >= latitude[0]) & (lat <= latitude[1]))[0]).transpose('time', 'lat', 'lon')
    yr, mon = da.time.dt.year.values, da.time.dt.month.values
    totals, n_days, years_out = [], [], []
    for y in range(YEARS[0], YEARS[1] + 1):                      # one season at a time (memory)
        idx = np.where((yr == y) & np.isin(mon, MONTHS))[0]
        if idx.size == 0:
            continue
        p = da.isel(time=idx).values * UNIT_FACTORS[units]       # mm/day
        totals.append(np.nansum(p, axis=0) / 1000)               # m
        n_days.append(idx.size)
        years_out.append(y)
    n_days = np.array(n_days)
    full = n_days >= n_days.max() - 2                            # drop incomplete seasons
    if not full.all():
        print(f'  {model}: incomplete seasons dropped: {np.array(years_out)[~full].tolist()}')
    out = xr.DataArray(np.array(totals)[full], dims=('year', 'lat', 'lon'),
                       coords={'year': np.array(years_out)[full], 'lat': da.lat.values, 'lon': da.lon.values},
                       name='pr_total', attrs={'units': 'm', 'months': '5-10', 'member': member,
                                               'days_per_season': int(n_days.max())})
    os.makedirs(CMIP6_CACHE_DIR, exist_ok=True)
    out.to_netcdf(cache + '.tmp')
    os.replace(cache + '.tmp', cache)
    return out


In [19]:
# members used in the analysis: first ssp370 member of each model in the precip_cover files
pc_pat = re.compile(r'^precip_cover_(?P<model>[^_]+)_ssp370_(?P<member>r\d+i\d+p\d+f\d+)_30-60N\.nc$')
members = {}
for f in os.listdir(CMIP6_PC_DIR):
    if (m := pc_pat.match(f)) and m['model'] not in CMIP6_EXCLUDE:
        members.setdefault(m['model'], []).append(m['member'])
members = {mod: sorted(mems, key=member_sort_key)[0] for mod, mems in sorted(members.items(), key=lambda kv: kv[0].lower())}

rows = []
for model, member in members.items():
    tot = yearly_totals(model, member)
    sftlf, mask_name = sftlf_on(model, tot.lat.values, tot.lon.values)
    w = xr.DataArray(np.cos(np.deg2rad(tot.lat.values))[:, None] * np.ones(tot.sizes['lon']),
                     dims=('lat', 'lon'), coords={'lat': tot.lat, 'lon': tot.lon})
    row = dict(model=model, member=member, grid=f"{tot.sizes['lat']}x{tot.sizes['lon']}")
    for region, sel in [('land', sftlf > 0), ('sea', sftlf == 0)]:
        wr = w.where(sel)                                               # cos(lat) weights, region cells only
        hist = tot.sel(year=slice(*PERIOD_HIST))
        fut = tot.sel(year=slice(*PERIOD_FUT))
        for p, d in [(PERIOD_HIST, hist), (PERIOD_FUT, fut)]:
            if d.sizes['year'] != p[1] - p[0] + 1:
                print(f'  WARNING {model}: {d.sizes["year"]} of {p[1] - p[0] + 1} years in {p}')
        h = float(hist.mean('year').weighted(wr.fillna(0)).mean(['lat', 'lon']))   # m, area-weighted mean
        f_ = float(fut.mean('year').weighted(wr.fillna(0)).mean(['lat', 'lon']))
        row.update({f'{region}_hist_m': h, f'{region}_fut_m': f_, f'{region}_diff_pct': (f_ - h) / h * 100})
    rows.append(row)
    print(f"{model:14s} {member:9s} land {row['land_diff_pct']:+6.2f}%  sea {row['sea_diff_pct']:+6.2f}%  (mask: {mask_name})")

cmip6 = pd.DataFrame(rows).set_index('model')


ACCESS-ESM1-5  r1i1p1f1  land  +1.07%  sea  +0.61%  (mask: sftlf_fx_ACCESS-ESM1-5_historical_r1i1p1f1_native.nc)
BCC-CSM2-MR    r1i1p1f1  land  +0.34%  sea  +1.03%  (mask: MPI-ESM1-2-HR (fallback))
CAMS-CSM1-0    r2i1p1f1  land  -0.79%  sea  -1.35%  (mask: sftlf_fx_CAMS-CSM1-0_1pctCO2_r2i1p1f1_native.nc)
CanESM5        r1i1p2f1  land  +2.05%  sea  +1.95%  (mask: sftlf_fx_CanESM5_historical_r1i1p1f1_native.nc)
CESM2          r1i1p1f1  land  -1.48%  sea  -0.54%  (mask: sftlf_fx_CESM2_historical_r1i1p1f1_native.nc)
CNRM-ESM2-1    r1i1p1f2  land  -0.41%  sea  -0.77%  (mask: sftlf_fx_CNRM-ESM2-1_historical_r2i1p1f2_native.nc)
EC-Earth3      r1i1p1f1  land  +3.18%  sea  +0.47%  (mask: sftlf_fx_EC-Earth3_historical_r1i1p1f1_native.nc)
FGOALS-g3      r1i1p1f1  land  +1.14%  sea  +1.90%  (mask: sftlf_fx_FGOALS-g3_historical_r1i1p1f1_native.nc)
GFDL-ESM4      r1i1p1f1  land  -1.07%  sea  +0.07%  (mask: sftlf_fx_GFDL-ESM4_historical_r1i1p1f1_native.nc)
INM-CM5-0      r1i1p1f1  land  +1.07%  sea  

In [20]:
# multi-model summary: each model has equal weight
summary = pd.DataFrame({
    stat: {col: fn(cmip6[col]) for col in ['land_diff_pct', 'sea_diff_pct']}
    for stat, fn in [('CMIP6 mean', np.mean), ('CMIP6 median', np.median), ('CMIP6 std', lambda s: s.std(ddof=1)),
                     ('CMIP6 min', np.min), ('CMIP6 max', np.max),
                     ('models > 0', lambda s: int((s > 0).sum())), ('models < 0', lambda s: int((s < 0).sum()))]
}).T
print(f'{len(cmip6)} CMIP6 models, May-Oct, {PERIOD_FUT[0]}-{PERIOD_FUT[1]} vs {PERIOD_HIST[0]}-{PERIOD_HIST[1]}')
print('land area mean precip diff for CMIP6 (mean of models) is ' + str(summary.loc['CMIP6 mean', 'land_diff_pct']) + '%')
print('sea area mean precip diff for CMIP6 (mean of models) is ' + str(summary.loc['CMIP6 mean', 'sea_diff_pct']) + '%')

os.makedirs(os.path.dirname(CMIP6_OUT_CSV), exist_ok=True)
pd.concat([cmip6, summary]).to_csv(CMIP6_OUT_CSV)
summary.round(2)


15 CMIP6 models, May-Oct, 2035-2049 vs 2021-2034
land area mean precip diff for CMIP6 (mean of models) is 0.29822775212873165%
sea area mean precip diff for CMIP6 (mean of models) is 0.3893077479990129%


,land_diff_pct,sea_diff_pct
CMIP6 mean,0.30,0.39
CMIP6 median,0.41,0.31
CMIP6 std,1.51,0.93
CMIP6 min,-2.87,-1.35
CMIP6 max,3.18,1.95
models > 0,9.00,11.00
models < 0,6.00,4.00


In [21]:
# all models and ensembles side by side (relative change in %, 2035-2049 vs 2021-2034)
compare = pd.DataFrame({
    'land_diff_pct': {'IFS cycle4': float(ifs_land_diff), 'ICON cycle4': float(icon_land_diff),
                      'CMIP6 (mean of models)': summary.loc['CMIP6 mean', 'land_diff_pct']},
    'sea_diff_pct': {'IFS cycle4': float(ifs_sea_diff), 'ICON cycle4': float(icon_sea_diff),
                     'CMIP6 (mean of models)': summary.loc['CMIP6 mean', 'sea_diff_pct']},
})
compare.round(2)


,land_diff_pct,sea_diff_pct
IFS cycle4,-3.01,2.19
ICON cycle4,-4.87,-0.41
CMIP6 (mean of models),0.30,0.39
